# Stage 1 — Data Exploration

Before we build anything, we need to understand the raw material.

StatsBomb provides free event data via the `statsbombpy` library. Event data means **one row per on-ball action** — every pass, shot, tackle, carry, and dribble, with x/y coordinates. This is very different from the tidy 'one row per player with 40 stats' table our ML model will eventually need. Understanding the shape and scope of this data now means Stage 2's feature engineering is grounded in reality.

**Questions we're answering in this notebook:**
1. What competitions and seasons are actually available to us?
2. How much data is there, and is it all usable?
3. What does the data hierarchy look like — competitions → matches → events?
4. What does a single match's event data actually contain?

In [1]:
import pandas as pd
import statsbombpy.sb as sb

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 1. What competitions do we have?

`sb.competitions()` returns the full catalogue of what StatsBomb makes available for free. Each row is a **competition-season pair** — so La Liga 2018/19 and La Liga 2019/20 are two separate rows.

The two columns that matter most for us:
- `match_available` — if this is non-null, standard event data exists for this competition-season. This is our core data source.
- `match_available_360` — richer spatial data (where every player on the pitch was during each event). Far fewer competitions have it; we don't need it for Stage 1.

In [2]:
competitions = sb.competitions()
print(f"Total competition-seasons: {len(competitions)}")
competitions[['competition_name', 'country_name', 'season_name', 'competition_gender']].sort_values(['competition_name', 'season_name'])

Total competition-seasons: 80


/Users/kareemel-wakeel/Football Analytics/football-scouting-tool/venv/lib/python3.13/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,competition_name,country_name,season_name,competition_gender
1,1. Bundesliga,Germany,2015/2016,male
0,1. Bundesliga,Germany,2023/2024,male
2,African Cup of Nations,Africa,2023,male
20,Champions League,Europe,1970/1971,male
19,Champions League,Europe,1971/1972,male
18,Champions League,Europe,1972/1973,male
17,Champions League,Europe,1999/2000,male
16,Champions League,Europe,2003/2004,male
15,Champions League,Europe,2004/2005,male
14,Champions League,Europe,2006/2007,male


## 2. Scope and availability check

Before assuming all 80 rows are usable, we should verify availability and understand the gender/youth split. These flags matter because:
- A similarity model trained on men's and women's data together doesn't make much sense — physical metrics are not comparable across genders.
- Youth competitions have a different player pool and tactical context than senior club football.

We'll likely filter to **senior male club football** for the main model, with the option to run a separate model on women's data.

In [3]:
print(f"Event data available:     {competitions['match_available'].notna().sum()} / {len(competitions)}")
print(f"360 data available:       {competitions['match_available_360'].notna().sum()} / {len(competitions)}")
print()
print(competitions['competition_gender'].value_counts().to_string())
print()
youth = competitions[competitions['competition_youth']]
print(f"Youth competitions ({len(youth)}): {youth['competition_name'].tolist()}")

Event data available:     80 / 80
360 data available:       12 / 80

competition_gender
male      67
female    13

Youth competitions (1): ['FIFA U20 World Cup']


## 3. The data hierarchy — competitions → matches → events

`sb.competitions()` told us *which* competition-seasons exist. The next level down is matches: given a competition and season, how many games do we have?

This matters because StatsBomb open data is **not necessarily a complete season** — they curate which matches to release. La Liga 2018/19 is a good test case: a well-known, richly tagged competition. We use `competition_id=11` (La Liga) and `season_id=4` (2018/19), which we confirmed from the competitions table.

A full La Liga season has 380 matches (20 teams × 19 home games each). If we see significantly fewer, that's an important constraint — it means our player sample per competition may be smaller than expected, and pooling across competitions becomes even more important.

In [4]:
matches = sb.matches(competition_id=11, season_id=4)
print(f"Matches in La Liga 2018/19: {len(matches)} (full season = 380)")
print(f"Columns: {matches.shape[1]}")
print()
matches[['match_id', 'match_date', 'home_team', 'away_team', 'home_score', 'away_score', 'match_week']].sort_values('match_week')

/Users/kareemel-wakeel/Football Analytics/football-scouting-tool/venv/lib/python3.13/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Matches in La Liga 2018/19: 34 (full season = 380)
Columns: 55



,match_id,match_date,home_team,away_team,home_score,away_score,match_week
12,15946,2018-08-18,Barcelona,Deportivo Alavés,3,0,1
32,15956,2018-08-25,Real Valladolid,Barcelona,0,1,2
21,15973,2018-09-02,Barcelona,Huesca,8,2,3
0,15978,2018-09-15,Real Sociedad,Barcelona,1,2,4
28,15986,2018-09-23,Barcelona,Girona,2,2,5
1,15998,2018-09-26,Leganés,Barcelona,2,1,6
27,16010,2018-09-29,Barcelona,Athletic Club,1,1,7
30,16023,2018-10-07,Valencia,Barcelona,1,1,8
2,16029,2018-10-20,Barcelona,Sevilla,4,2,9
16,16056,2018-11-11,Barcelona,Real Betis,3,4,12


## 4. The event level — what does a single match actually contain?

The matches table gave us metadata: who played, what score, which week. But the real data lives one level deeper: **events**.

Each row in the events table is one on-ball action — a pass, shot, tackle, carry, dribble, or any other interaction with the ball. This is the raw material our feature engineering in Stage 2 will aggregate into per-player profiles.

Before we can design those features, we need to understand what event types exist and what information is captured for each one.

In [5]:
match_id = matches['match_id'].iloc[0]
events = sb.events(match_id=match_id)
print(f"Events in this match: {len(events)}")
print(f"Columns: {events.shape[1]}")
print()
events['type'].value_counts()

/Users/kareemel-wakeel/Football Analytics/football-scouting-tool/venv/lib/python3.13/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Events in this match: 3905
Columns: 89



type
Pass              1148
Ball Receipt*     1044
Carry              989
Pressure           253
Ball Recovery      104
Duel                65
Block               34
Dribble             32
Interception        31
Foul Committed      28
Foul Won            27
Clearance           25
Dispossessed        22
Miscontrol          22
Goal Keeper         22
Shot                20
Dribbled Past       17
Substitution         6
Half Start           4
Half End             4
Tactical Shift       4
Starting XI          2
Shield               1
Bad Behaviour        1
Name: count, dtype: int64

### Event sub-attributes

The 89 columns are not all populated for every event — each event type has its own specific fields. A pass has `pass_length`, `pass_angle`, `pass_outcome`; a shot has `shot_statsbomb_xg`, `shot_outcome`. Most of these columns are null for events of other types.

Filtering to just pass events and dropping empty columns shows us exactly what we have to work with when building passing metrics.

In [6]:
passes = events[events['type'] == 'Pass']
pass_cols = passes.dropna(axis=1, how='all').columns.tolist()
print(f"Columns populated for pass events: {len(pass_cols)}")
print(pass_cols)

Columns populated for pass events: 44
['counterpress', 'duration', 'id', 'index', 'location', 'match_id', 'minute', 'off_camera', 'pass_aerial_won', 'pass_angle', 'pass_assisted_shot_id', 'pass_body_part', 'pass_cross', 'pass_cut_back', 'pass_end_location', 'pass_goal_assist', 'pass_height', 'pass_inswinging', 'pass_length', 'pass_outcome', 'pass_outswinging', 'pass_recipient', 'pass_recipient_id', 'pass_shot_assist', 'pass_straight', 'pass_switch', 'pass_technique', 'pass_through_ball', 'pass_type', 'period', 'play_pattern', 'player', 'player_id', 'position', 'possession', 'possession_team', 'possession_team_id', 'related_events', 'second', 'team', 'team_id', 'timestamp', 'type', 'under_pressure']


## 5. How many matches do other leagues have — and who actually has enough data?

The Barcelona finding isn't unique to La Liga 2018/19. Let's check the most recent seasons across three major leagues to confirm the pattern.

In [7]:
leagues = [
    ('La Liga',        11,  90, '2020/2021', 380),
    ('Ligue 1',         7, 235, '2022/2023', 380),
    ('1. Bundesliga',   9, 281, '2023/2024', 306),
]

for name, cid, sid, season, full_season in leagues:
    matches = sb.matches(competition_id=cid, season_id=sid)
    teams = sorted(set(matches['home_team'].tolist() + matches['away_team'].tolist()))
    print(f"{name} {season}: {len(matches)} matches (full season = {full_season})")
    print(f"  {len(teams)} teams appear: {teams}")
    print()

La Liga 2020/2021: 35 matches (full season = 380)
  19 teams appear: ['Athletic Club', 'Atlético Madrid', 'Barcelona', 'Celta Vigo', 'Cádiz', 'Deportivo Alavés', 'Elche', 'Getafe', 'Granada', 'Huesca', 'Levante UD', 'Osasuna', 'Real Betis', 'Real Madrid', 'Real Sociedad', 'Real Valladolid', 'Sevilla', 'Valencia', 'Villarreal']

Ligue 1 2022/2023: 32 matches (full season = 380)
  20 teams appear: ['AC Ajaccio', 'AS Monaco', 'Angers', 'Auxerre', 'Clermont Foot', 'Lens', 'Lille', 'Lorient', 'Lyon', 'Montpellier', 'Nantes', 'OGC Nice', 'Olympique de Marseille', 'Paris Saint-Germain', 'Rennes', 'Stade Brestois', 'Stade de Reims', 'Strasbourg', 'Toulouse', 'Troyes']



/Users/kareemel-wakeel/Football Analytics/football-scouting-tool/venv/lib/python3.13/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


1. Bundesliga 2023/2024: 34 matches (full season = 306)
  18 teams appear: ['Augsburg', 'Bayer Leverkusen', 'Bayern Munich', 'Bochum', 'Borussia Dortmund', 'Borussia Mönchengladbach', 'Darmstadt 98', 'Eintracht Frankfurt', 'FC Heidenheim', 'FC Köln', 'FSV Mainz 05', 'Freiburg', 'Hoffenheim', 'RB Leipzig', 'Union Berlin', 'VfB Stuttgart', 'Werder Bremen', 'Wolfsburg']



### What this means for our player pool

The pattern is consistent: **each competition-season is one team's complete fixture list**, not a full season. La Liga 2020/21 is Barcelona's 35 games, Bundesliga 2023/24 is Bayern Munich's 34 games, and so on.

Every other team in the league appears as an opponent — but only in the 2 matches they played against the featured club (once home, once away). That's roughly 180 minutes per opponent player, which will almost certainly fall below the minimum-minutes threshold we'll set in Stage 2.

**The practical consequence:**

- **Featured club players** (Barcelona, Bayern, PSG, etc.) — have a full season of data from that competition alone. Reliably profileable.
- **Opponent players** — have ~180 minutes from that season. On their own, not enough. But they may accumulate minutes across other competition-seasons in the dataset (e.g. a Champions League season also featuring their club, or a World Cup, or another La Liga season where they faced a different featured team).

**The honest bias to acknowledge:** our model will naturally be richer for players at top clubs that StatsBomb chose to feature. A Barcelona squad player with 900 minutes will have a far more reliable profile than a Huesca midfielder who appeared in 2 matches across the entire dataset. This isn't a flaw we can fix — it's a constraint of the open data. We handle it by setting a minimum-minutes threshold and being transparent about who falls below it.

---

## Stage 1 Summary — What we learned

### The data universe
- **80 competition-seasons** available, all with standard event data
- **67 male, 13 female** — we'll build the main model on male competitions only; metrics aren't comparable across genders
- **1 youth competition** (FIFA U20 World Cup) — excluded; different player pool and tactical context
- **12 of 80** have 360 data (every player's position on the pitch) — richer, but too narrow a subset to build on

### The open data is curated, not complete
StatsBomb open data is released as **one team's full fixture list per competition-season**, not a random sample. La Liga 2018/19 is exclusively Barcelona's 34 matches — not the full 380-game season. Every opponent Barcelona faced appears in the data, but bottom-half clubs who never played them are invisible.

**Consequence:** pooling across all 80 competition-seasons is essential. We cannot build a reliable player profile from a single competition.

### The data hierarchy
```
competitions  →  matches  →  events
(80 rows)        (~34/comp)   (~3,900/match, 89 columns)
```

### What event data gives us
Each match contains ~3,900 on-ball actions. The most common types:

| Event type | Count (1 match) | Value for profiling |
|---|---|---|
| Pass | ~1,150 | Very high — completion rate, progressive passes, key passes, crosses |
| Carry | ~990 | High — progressive carrying, distance |
| Pressure | ~250 | High — pressing intensity |
| Duel | ~65 | Medium — aerial and ground duel win rates |
| Shot | ~20 | High but sparse — xG, shot volume |
| Dribble | ~30 | Medium — success rate |

### Key quirks to remember in Stage 2
1. **`pass_outcome` NaN = success.** StatsBomb only records an outcome when something goes wrong. Treat null as completed.
2. **Progressive passes need coordinate math.** Use `location` and `pass_end_location` to calculate whether a pass moved the ball toward the opponent's goal — it's not a pre-computed column.
3. **Set pieces vs open play.** `pass_type` flags corners, free kicks, and throw-ins. These should be separated from open-play passing metrics to avoid skewing profiles.
4. **Minutes played.** `Starting XI` and `Substitution` events tell us how long each player was on the pitch — essential for normalising everything to per-90 stats.
5. **Small sample sizes.** A player appearing in 1–2 matches may have fewer than 5 shots. We'll need a minimum-minutes threshold in Stage 2 to exclude unreliable profiles.